In [1]:
# Find the project root from either the workspace or notebooks folder.
from pathlib import Path
from datetime import datetime, timezone
import json
import sys
PROJECT_ROOT = next((folder for folder in (Path.cwd(), *Path.cwd().parents)
                     if (folder / "src/task_b").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Open the NLP_PROJECT_task_b folder in VS Code before running this notebook")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# Official FEVER inputs (never downloaded here); prepared data is rebuilt from them, and each run gets a new folder.
from task_b.fever import claim_key, load_source_prior, prepare_fever, write_prepared, write_validation_sample
from task_b.verification import (choose_threshold, evaluate, fever_binary_proxy, lexical_cues, lexical_overlap,
                                 majority_prior, save_run, timed_predictions)
TRAIN_CLAIMS = PROJECT_ROOT / "data/raw/fever/train.jsonl"
EVAL_CLAIMS = PROJECT_ROOT / "data/raw/fever/shared_task_dev.jsonl"
WIKI_DIR = PROJECT_ROOT / "data/raw/fever/wiki-pages"
SEED, VALIDATION_PER_LABEL = 42, 2000
PREPARED = PROJECT_ROOT / "outputs/task_b/fever_prepared"
RUN_DIR = PROJECT_ROOT / "outputs/task_b/fever_baselines" / datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%SZ")
if TRAIN_CLAIMS.resolve() == EVAL_CLAIMS.resolve():
    raise ValueError("Training and evaluation files must differ")
missing = [str(path) for path in (TRAIN_CLAIMS, EVAL_CLAIMS, WIKI_DIR) if not path.exists()]
if missing:
    raise FileNotFoundError("Provide the official FEVER inputs: " + ", ".join(missing))

In [2]:
# Test set: resolve gold evidence for the labelled shared-task dev claims. NEI claims have no gold evidence,
# so they are excluded and counted; claims whose evidence cannot be found are reported, never filled in.
print("Resolving test evidence; the Wikipedia scan takes several minutes.")
test_records, test_report = prepare_fever(EVAL_CLAIMS, WIKI_DIR)
write_prepared(test_records, test_report, PREPARED / "test")
print("test:", {key: test_report[key] for key in ("resolved", "unresolved", "skipped_nei")})
assert test_records, "No gold-evidence records resolved; inspect resolution_report.json"

Resolving test evidence; the Wikipedia scan takes several minutes.


test: {'resolved': 13202, 'unresolved': 130, 'skipped_nei': 6666}


In [3]:
# Validation set: a seeded, balanced sample of SUPPORTS/REFUTES training claims (none shares its text with a
# test claim), used only to choose thresholds. Balanced because the test claims are balanced.
test_claim_keys = {claim_key(json.loads(line)["claim"]) for line in EVAL_CLAIMS.read_text(encoding="utf-8").splitlines() if line.strip()}
validation_dir = PREPARED / f"validation_seed{SEED}_n{VALIDATION_PER_LABEL}"
manifest = write_validation_sample(TRAIN_CLAIMS, validation_dir / "claims.jsonl", VALIDATION_PER_LABEL, SEED, test_claim_keys)
print("Resolving validation evidence; this needs a second Wikipedia scan.")
validation_records, validation_report = prepare_fever(validation_dir / "claims.jsonl", WIKI_DIR)
write_prepared(validation_records, validation_report, validation_dir)
(validation_dir / "sample_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("validation:", {"sampled": manifest["rows"], "resolved": validation_report["resolved"],
                      "unresolved": validation_report["unresolved"],
                      "training claims skipped (text also in test)": manifest["skipped_overlapping_with_evaluation"]})

Resolving validation evidence; this needs a second Wikipedia scan.


validation: {'sampled': 4000, 'resolved': 3965, 'unresolved': 35, 'training claims skipped (text also in test)': 3}


In [4]:
# Majority prior from the training labels (SUPPORTS/REFUTES only); evaluation labels are never used to fit it.
prior = majority_prior(load_source_prior(TRAIN_CLAIMS))
print("Training prior:", prior)

Training prior: SUPPORTED


In [5]:
# Choose each lexical baseline's overlap cutoff by validation macro-F1 (two-class diagnostic); test is not used.
GRID = [round(0.1 * step, 1) for step in range(1, 10)]
validation_examples = [row.claim for row in validation_records]
validation_gold = [row.gold_label for row in validation_records]
thresholds, searches = {}, {}
for name, function in [("lexical", lexical_overlap), ("lexical_cues", lexical_cues)]:
    thresholds[name], searches[name] = choose_threshold(
        validation_gold, lambda cutoff, f=function: [f(example, cutoff) for example in validation_examples], GRID)
    print(f"{name:<13} chosen cutoff {thresholds[name]} | validation macro-F1:",
          " ".join(f"{row['threshold']}={row['validation_macro_f1']:.3f}" for row in searches[name]))

lexical       chosen cutoff 0.6 | validation macro-F1: 0.1=0.353 0.2=0.385 0.3=0.444 0.4=0.496 0.5=0.540 0.6=0.569 0.7=0.562 0.8=0.538 0.9=0.455


lexical_cues  chosen cutoff 0.5 | validation macro-F1: 0.1=0.556 0.2=0.568 0.3=0.590 0.4=0.608 0.5=0.621 0.6=0.613 0.7=0.582 0.8=0.549 0.9=0.453


In [6]:
# Score the test set once: majority prior, and each lexical baseline at the original fixed 0.5 cutoff and at its
# validation-chosen cutoff. UNSUPPORTED counts as CONTRADICTED only for this two-class scoring; raw labels are saved.
examples = [row.claim for row in test_records]
gold = [row.gold_label for row in test_records]
methods = {"majority": (lambda example: prior, None)}
for name, function in [("lexical", lexical_overlap), ("lexical_cues", lexical_cues)]:
    methods[f"{name}_fixed_0.5"] = (lambda example, f=function: f(example, 0.5), 0.5)
    methods[f"{name}_validation"] = (lambda example, f=function, t=thresholds[name]: f(example, t), thresholds[name])
results = {}
for name, (predictor, cutoff) in methods.items():
    predictions, latency = timed_predictions(examples, predictor)
    binary = [fever_binary_proxy(label) for label in predictions]
    score = evaluate(gold, binary, ("SUPPORTED", "CONTRADICTED"), latency)
    score["raw_unsupported_count"] = predictions.count("UNSUPPORTED")
    score["evaluation_scope"] = "FEVER SUPPORTS/REFUTES gold-evidence diagnostic (NEI excluded)"
    results[name] = score
    config = {"method": name, "threshold": cutoff, "threshold_source": "validation" if name.endswith("_validation") else "fixed",
              "validation_search": searches.get(name.rsplit("_", 1)[0]) if name.endswith("_validation") else None,
              "validation_manifest": manifest, "prior_source": str(TRAIN_CLAIMS), "evaluation_source": str(EVAL_CLAIMS),
              "test_resolution": {key: test_report[key] for key in ("resolved", "unresolved", "skipped_nei")},
              "binary_proxy": "UNSUPPORTED -> CONTRADICTED for this diagnostic only"}
    save_run(RUN_DIR / name, config, [{"claim_id": row.claim.claim_id, "gold": row.gold_label, "raw_prediction": prediction,
                                       "binary_prediction": projected, "latency_seconds": elapsed}
                                      for row, prediction, projected, elapsed in zip(test_records, predictions, binary, latency)], score)
print("Saved to", RUN_DIR.relative_to(PROJECT_ROOT))

Saved to outputs\task_b\fever_baselines\run_20261022T010816Z


In [7]:
# Compact test scores, then per-class scores and two-class confusion matrices (gold rows, predicted columns).
print(f"{'method':<24} {'n':>6} {'accuracy':>9} {'macro-F1':>9} {'wrong SUPPORTED':>16} {'raw UNSUPPORTED':>16} {'mean ms':>8}")
for name, score in results.items():
    print(f"{name:<24} {score['n']:>6} {score['accuracy']:>9.3f} {score['macro_f1']:>9.3f} {score['incorrect_supported']:>16} "
          f"{score['raw_unsupported_count']:>16} {score['latency_mean_ms']:>8.3f}")
for name, score in results.items():
    print(f"\n{name}")
    for label, values in score["per_class"].items():
        print(f"  {label:<13} precision {values['precision']:.3f}  recall {values['recall']:.3f}  F1 {values['f1']:.3f}  support {values['support']}")
    for label, counts in zip(score["labels"], score["confusion_matrix"]):
        print(f"  gold {label:<13} predicted SUPPORTED {counts[0]:>6}  CONTRADICTED {counts[1]:>6}")

method                        n  accuracy  macro-F1  wrong SUPPORTED  raw UNSUPPORTED  mean ms
majority                  13202     0.500     0.333             6605                0    0.000
lexical_fixed_0.5         13202     0.551     0.524             4538             3456    0.015
lexical_validation        13202     0.575     0.574             3225             5760    0.015
lexical_cues_fixed_0.5    13202     0.602     0.593             3630             3432    0.031
lexical_cues_validation   13202     0.602     0.593             3630             3432    0.029

majority
  SUPPORTED     precision 0.500  recall 1.000  F1 0.666  support 6597
  CONTRADICTED  precision 0.000  recall 0.000  F1 0.000  support 6605
  gold SUPPORTED     predicted SUPPORTED   6597  CONTRADICTED      0
  gold CONTRADICTED  predicted SUPPORTED   6605  CONTRADICTED      0

lexical_fixed_0.5
  SUPPORTED     precision 0.534  recall 0.789  F1 0.637  support 6597
  CONTRADICTED  precision 0.598  recall 0.313  F1 0.4

In [8]:
# Summary from the measured scores, and what this diagnostic does not measure.
best = max(results, key=lambda name: results[name]["macro_f1"])
print(f"Highest test macro-F1: {best} ({results[best]['macro_f1']:.3f}) on {results[best]['n']:,} resolved SUPPORTS/REFUTES claims.")
print("Limits: NEI claims are excluded, so this is not the three-class Task B evaluation; gold evidence is given, "
      "so retrieval is not tested; CONTRADICTED scores include UNSUPPORTED predictions counted as not-supported.")

Highest test macro-F1: lexical_cues_fixed_0.5 (0.593) on 13,202 resolved SUPPORTS/REFUTES claims.
Limits: NEI claims are excluded, so this is not the three-class Task B evaluation; gold evidence is given, so retrieval is not tested; CONTRADICTED scores include UNSUPPORTED predictions counted as not-supported.
